# Ejecucion de random forest

In [3]:
%pip install pandas numpy scikit-learn


Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'c:\Users\aleja\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


## Generación del modelo

In [6]:
def generate_model_rf(X_train, X_val, y_train, y_val, n_estimators, max_depth, min_samples_split):
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    r2 = r2_score(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)

    return r2, mae, mse, rmse

## Entrenamiento principal para entrenar y comparar combinaciones de hiperparámetros

In [17]:
def train_model_with_hyperparameters():
    # Cargar archivos
    df_train = pd.read_csv('../Data/train.csv')
    df_val = pd.read_csv('../Data/validation.csv')
    df_test = pd.read_csv('../Data/test.csv')

    # Combinar train + validation para codificar categóricas de forma uniforme
    df_combined = pd.concat([df_train, df_val])
    label_encoders = {}
    for col in df_combined.select_dtypes(include='object').columns:
        le = LabelEncoder()
        df_combined[col] = le.fit_transform(df_combined[col].astype(str))
        label_encoders[col] = le

    # Repartir codificados
    df_train = df_combined.iloc[:len(df_train)].copy()
    df_val = df_combined.iloc[len(df_train):].copy()

    # Aplicar la misma codificación a test
    for col in label_encoders:
        df_test[col] = label_encoders[col].transform(df_test[col].astype(str))

    # Definir X e y
    X_train = df_train.drop(columns=['G1', 'G2', 'G3'])
    y_train = df_train['G3']
    X_val = df_val.drop(columns=['G1', 'G2', 'G3'])
    y_val = df_val['G3']

    # Combinaciones de hiperparámetros
    n_estimators_list = [100, 200]
    max_depth_list = [10, 20]
    min_samples_split_list = [2, 4]

    # Evaluación de combinaciones
    for n in n_estimators_list:
        for d in max_depth_list:
            for s in min_samples_split_list:
                r2, mae, mse, rmse = generate_model_rf(X_train, X_val, y_train, y_val, n, d, s)
                print(f"Combinación: n_estimators={n}, max_depth={d}, min_samples_split={s}")
                print(f"R²: {r2:.4f} | MAE: {mae:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f}")
                print('-' * 70)


In [18]:
train_model_with_hyperparameters()

Combinación: n_estimators=100, max_depth=10, min_samples_split=2
R²: 0.3125 | MAE: 3.1934 | MSE: 15.5066 | RMSE: 3.9378
----------------------------------------------------------------------
Combinación: n_estimators=100, max_depth=10, min_samples_split=4
R²: 0.3001 | MAE: 3.1946 | MSE: 15.7868 | RMSE: 3.9733
----------------------------------------------------------------------
Combinación: n_estimators=100, max_depth=20, min_samples_split=2
R²: 0.3162 | MAE: 3.1768 | MSE: 15.4216 | RMSE: 3.9270
----------------------------------------------------------------------
Combinación: n_estimators=100, max_depth=20, min_samples_split=4
R²: 0.3023 | MAE: 3.1988 | MSE: 15.7353 | RMSE: 3.9668
----------------------------------------------------------------------
Combinación: n_estimators=200, max_depth=10, min_samples_split=2
R²: 0.3194 | MAE: 3.2056 | MSE: 15.3502 | RMSE: 3.9179
----------------------------------------------------------------------
Combinación: n_estimators=200, max_depth=10, 